# MOUSE · Multimodální embeddingy v1 · KAGGLE
## Tři vybrané pohádky a jejich ilustrace

Tato verze je připravená pro Kaggle Notebooks. Používá GPU, pokud ho v nastavení notebooku zapneš.

**Experiment:** Myš, Kocour a Ryba. Každá karta se zpracuje ve třech režimech:
1. původní česká pohádka,
2. odpovídající ilustrace,
3. pohádka a ilustrace společně.

Notebook načte soubory přímo z veřejného repozitáře GitHub. Model Qwen3-VL-Embedding-2B se při prvním běhu stáhne z Hugging Face. Výsledky se uloží do `/kaggle/working` a do ZIP archivu.

**Důležité:** V Kaggle nejprve vpravo v Settings nastav Accelerator → GPU a zapni Internet. Model má několik GB, takže první spuštění může chvíli trvat.

Toto je embeddingový experiment. Výsledky nejsou automatickým měřením 12 úhlů MOUSE ani lidsky ověřeným měřením významu.

In [ ]:
# 1. Instalace potřebných knihoven pro Kaggle
# Kaggle má většinu vědeckých knihoven předinstalovanou.
# Doinstalujeme pouze balíčky potřebné pro Qwen3-VL-Embedding.
import importlib.metadata as metadata
import subprocess, sys

required = {
    "sentence-transformers": ("sentence-transformers", (5, 4, 0)),
    "transformers": ("transformers", (4, 57, 1)),
}

def version_tuple(value):
    parts = []
    for part in value.split(".")[:3]:
        digits = "".join(ch for ch in part if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
    return tuple(parts)

to_install = []
for package, (distribution, minimum) in required.items():
    try:
        installed = metadata.version(distribution)
        if version_tuple(installed) < minimum:
            to_install.append(f"{package}>={'.'.join(map(str, minimum))}")
            print(f"{package}: {installed} → aktualizuji")
        else:
            print(f"{package}: {installed} · vyhovuje")
    except metadata.PackageNotFoundError:
        to_install.append(f"{package}>={'.'.join(map(str, minimum))}")

if to_install:
    print("Instaluji:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
else:
    print("Potřebné balíčky jsou připravené.")

In [ ]:
# 2. Importy a nastavení
import os, io, re, json, time, zipfile, requests, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

warnings.filterwarnings("ignore")

REPO = "https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main"
TEXT_DIR = "doma/primarni_data/kriva_pohadka"
IMAGE_DIR = "doma/primarni_data/ilustrace"
MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"

# Při prvním běhu může načítání modelu chvíli trvat.
# Model se stáhne do úložiště Kaggle.
OUTPUT_DIR = Path("/kaggle/working/MOUSE_Multimodal_Embeddings_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_ID)
print("Výstupní složka Kaggle:", OUTPUT_DIR)

## 3. Tři vybrané karty

Používáme tři zvířata z okruhu pohádky o myši: **myš, kocour a ryba**.

U každého textu notebook přeskočí technické řádky s údaji o kartě a ilustraci. Začíná až skutečným nadpisem pohádky, například „O KOCOUROVI“. U myši přeskočí i obecný úvodní text před vlastním nadpisem „O MYŠI“.

Text pohádky se dál nijak nepřepisuje ani nezkracuje.

In [ ]:
# 3. Tři karty: myš, kocour a ryba
# Předloha: uživatelčin výběr, nikoli automatický výběr všech souborů.
# Každý text začíná až vlastním názvem pohádky. Technické hlavičky
# typu "Karta: ...", "Text: ..." a obecný úvodní rámec se vynechají.

CARD_FILES = [
    ("mys", "mys", "O MYŠI"),
    ("kocour", "kocour", "O KOCOUROVI"),
    ("ryba", "ryba", "O RYBĚ"),
]

def download_bytes(url):
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    return response.content

def extract_story_from_title(full_text, title_marker):
    lines = full_text.replace("\ufeff", "").splitlines()
    marker = title_marker.casefold()
    start = None
    for i, line in enumerate(lines):
        if line.strip().casefold().startswith(marker):
            start = i
            break
    if start is None:
        raise ValueError(f"Nenalezen nadpis pohádky: {title_marker}")
    story = "\n".join(lines[start:]).strip()
    if not story:
        raise ValueError(f"Za nadpisem {title_marker} není text.")
    return story

records = []
errors = []

for image_stem, text_stem, title_marker in CARD_FILES:
    text_url = f"{REPO}/{TEXT_DIR}/{text_stem}.txt"
    image_url = f"{REPO}/{IMAGE_DIR}/{image_stem}.jpg"
    try:
        raw_text = download_bytes(text_url).decode("utf-8-sig")
        story_text = extract_story_from_title(raw_text, title_marker)
        image_content = download_bytes(image_url)
        image = Image.open(io.BytesIO(image_content)).convert("RGB")
        records.append({
            "id": image_stem,
            "title": title_marker,
            "text_file": f"{text_stem}.txt",
            "image_file": f"{image_stem}.jpg",
            "text": story_text,
            "image": image,
            "image_bytes": image_content,
            "text_url": text_url,
            "image_url": image_url,
        })
        print(f"✓ {image_stem}: začátek = {story_text.splitlines()[0]}, "
              f"{len(story_text)} znaků, obrázek {image.size}")
    except Exception as e:
        errors.append({"id": image_stem, "error": str(e)})
        print(f"CHYBA {image_stem}: {e}")

print("\nNačteno dvojic:", len(records))
print("Chyby:", len(errors))
if errors:
    display(pd.DataFrame(errors))
if len(records) != len(CARD_FILES):
    raise RuntimeError("Některá z tří požadovaných dvojic se nenačetla. Nejdřív oprav chyby.");

## 4. Rychlá kontrola načtených karet

Než se spustí model, zobrazíme skutečné dvojice. Tím se dá odhalit špatné spárování nebo chybějící ilustrace.

In [ ]:
# 4. Náhled karet
from IPython.display import display, HTML

for r in records:
    print(f"\n{r['id']} · {r['text_file']} · {r['image_file']}")
    display(r["image"].resize((260, 260)))
    preview = r["text"][:650].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML(f"<div style='max-width:850px;white-space:pre-wrap'>{preview}{' …' if len(r['text'])>650 else ''}</div>"))

## 5. Načtení modelu

Qwen3-VL-Embedding-2B umí přijmout text, obrázek i jejich kombinaci. Použijeme stejný model a stejné výchozí nastavení pro všechny tři režimy.

To je důležité: nevytváříme textové vektory jedním modelem a obrazové jiným modelem, jejichž souřadnicové prostory by nemusely být srovnatelné.

In [ ]:
# 5. Načtení Qwen modelu
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Zařízení:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("VAROVÁNÍ: Kaggle notebook neběží na GPU. Zastav běh a zapni GPU v Settings → Accelerator.")
if device == "cpu":
    print("POZOR: Bez GPU bude výpočet výrazně pomalejší. V Kaggle otevři Settings → Accelerator → GPU.")

model = SentenceTransformer(
    MODEL_ID,
    device=device,
    trust_remote_code=True
)
print("Model načten.")

## 6. Tři embeddingy pro každou kartu

Pro každou kartu vytvoříme:
- `text`: pouze původní česká pohádka;
- `image`: pouze odpovídající ilustrace;
- `joint`: původní česká pohádka a ilustrace společně.

Nepřidáváme žádný popis obrázku vytvořený námi. Tím by se do experimentu dostala další interpretace, kterou zatím nechceme.

In [ ]:
# 6. Výpočet embeddingů
texts = [{"text": r["text"]} for r in records]
images = [{"image": r["image"]} for r in records]
joint = [{"text": r["text"], "image": r["image"]} for r in records]

def encode_in_batches(items, batch_size=1, label=""):
    chunks = []
    for start in range(0, len(items), batch_size):
        batch = items[start:start+batch_size]
        vec = model.encode(
            batch,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        chunks.append(vec)
        print(f"{label}: {min(start+batch_size, len(items))}/{len(items)}")
    return np.vstack(chunks)

# Batch 1 je záměrně konzervativní kvůli paměti GPU u obrázků.
text_vectors = encode_in_batches(texts, batch_size=1, label="TEXT")
image_vectors = encode_in_batches(images, batch_size=1, label="OBRAZ")
joint_vectors = encode_in_batches(joint, batch_size=1, label="TEXT + OBRAZ")

print("\nRozměry:")
print("Text:", text_vectors.shape)
print("Obraz:", image_vectors.shape)
print("Společně:", joint_vectors.shape)

assert text_vectors.shape == image_vectors.shape == joint_vectors.shape
assert np.isfinite(text_vectors).all() and np.isfinite(image_vectors).all() and np.isfinite(joint_vectors).all()
print("Kontrola rozměrů a čísel: OK")

## 7. Uložení surových vektorů

Tohle je důležitý výstup. Vektory nebudou jen jednorázové obrázky grafů. Uložíme je, abychom je mohli později načíst do dalších nástrojů MOUSE a zkoumat jinými způsoby.

In [ ]:
# 7. Uložení vektorů a metadat
ids = [r["id"] for r in records]
np.savez_compressed(
    OUTPUT_DIR / "embeddings.npz",
    ids=np.array(ids),
    text=text_vectors,
    image=image_vectors,
    joint=joint_vectors
)

metadata = []
for r in records:
    metadata.append({
        "id": r["id"],
        "text_file": r["text_file"],
        "image_file": r["image_file"],
        "text_url": r["text_url"],
        "image_url": r["image_url"],
        "text_characters": len(r["text"]),
        "text": r["text"],
    })

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Uloženo:", OUTPUT_DIR / "embeddings.npz")
print("Uloženo:", OUTPUT_DIR / "metadata.json")

## 8. Text proti ilustracím

Tady je jeden z hlavních pohledů. Každý řádek je text jedné pohádky, každý sloupec je ilustrace. Číslo je kosinová podobnost textového a obrazového embeddingu.

**Diagonála** (Ryba–Ryba, Myš–Myš atd.) ukazuje správně spárované dvojice. Ostatní políčka jsou kontrolní porovnání s ilustracemi jiných karet.

Samotná vysoká podobnost ještě neznamená, že obrázek „správně ilustruje“ celý příběh. Ukazuje jen blízkost v reprezentaci modelu.

In [ ]:
# 8. Křížová podobnost: text × obraz
text_image_sim = cosine_similarity(text_vectors, image_vectors)
sim_df = pd.DataFrame(text_image_sim, index=ids, columns=ids)

plt.figure(figsize=(max(12, len(ids)*0.38), max(9, len(ids)*0.32)))
sns.heatmap(sim_df, cmap="viridis", center=None)
plt.title("Qwen · podobnost českých textů a ilustrací")
plt.xlabel("Ilustrace")
plt.ylabel("Text pohádky")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

sim_df.to_csv(OUTPUT_DIR / "similarity_text_image.csv", encoding="utf-8-sig")

In [ ]:
# 9. Je vlastní ilustrace nejbližší?
# Pro každý text zjistíme pořadí jeho správné ilustrace mezi všemi ilustracemi.
rows = []
for i, card_id in enumerate(ids):
    scores = text_image_sim[i]
    order = np.argsort(scores)[::-1]
    rank = int(np.where(order == i)[0][0]) + 1
    rows.append({
        "karta": card_id,
        "podobnost_vlastni_ilustrace": float(scores[i]),
        "poradi_vlastni_ilustrace": rank,
        "nejpodobnejsi_ilustrace": ids[int(order[0])],
        "skore_nejpodobnejsi": float(scores[order[0]]),
        "rozdil_vlastni_minus_druha": float(scores[i] - scores[order[1]]) if len(order)>1 else np.nan,
    })

pairing_df = pd.DataFrame(rows).sort_values("poradi_vlastni_ilustrace")
display(pairing_df)
pairing_df.to_csv(OUTPUT_DIR / "text_image_pairing_ranks.csv", index=False, encoding="utf-8-sig")

print("Počet případů, kdy je vlastní ilustrace na 1. místě:",
      int((pairing_df["poradi_vlastni_ilustrace"] == 1).sum()), "/", len(pairing_df))

## 10. Podobnosti uvnitř každého typu reprezentace

Tyto tři matice odpovídají třem různým otázkám:
- Které pohádky jsou si podle modelu podobné textově?
- Které ilustrace jsou si podobné obrazově?
- Které karty jsou si podobné, když model dostane obojí společně?

Matice jsou v rámci stejného modelu a stejného prostoru. Nezaměňujeme je za lidské hodnocení významu.

In [ ]:
# 10. Tři matice podobností
text_sim = cosine_similarity(text_vectors)
image_sim = cosine_similarity(image_vectors)
joint_sim = cosine_similarity(joint_vectors)

def plot_matrix(matrix, title, filename):
    df = pd.DataFrame(matrix, index=ids, columns=ids)
    plt.figure(figsize=(max(10, len(ids)*0.34), max(8, len(ids)*0.30)))
    sns.heatmap(df, cmap="mako", vmin=-1, vmax=1)
    plt.title(title)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    df.to_csv(OUTPUT_DIR / filename, encoding="utf-8-sig")
    return df

text_sim_df = plot_matrix(text_sim, "TEXT · podobnosti mezi pohádkami", "similarity_text_text.csv")
image_sim_df = plot_matrix(image_sim, "OBRAZ · podobnosti mezi ilustracemi", "similarity_image_image.csv")
joint_sim_df = plot_matrix(joint_sim, "SPOLEČNĚ · podobnosti mezi kartami", "similarity_joint_joint.csv")

## 11. Co se změní, když model dostane text a obraz společně?

Společný embedding porovnáme se samostatným textovým a obrazovým embeddingem stejné karty. Výsledky ukazují, ke kterému samostatnému vstupu je společná reprezentace v prostoru modelu blíž.

To není přímé měření „kolik nového významu vzniklo“. Je to měřitelná změna reprezentace při společném vstupu.

In [ ]:
# 11. Vztah společné reprezentace k samostatnému textu a obrazu
joint_to_text = np.sum(joint_vectors * text_vectors, axis=1)
joint_to_image = np.sum(joint_vectors * image_vectors, axis=1)
text_to_image = np.sum(text_vectors * image_vectors, axis=1)

fusion_df = pd.DataFrame({
    "karta": ids,
    "text_vs_obraz": text_to_image,
    "spolecne_vs_text": joint_to_text,
    "spolecne_vs_obraz": joint_to_image,
    "rozdil_spolecne_text_minus_obraz": joint_to_text - joint_to_image,
})
display(fusion_df.round(4))
fusion_df.to_csv(OUTPUT_DIR / "joint_representation_comparison.csv", index=False, encoding="utf-8-sig")

ax = fusion_df.set_index("karta")[["spolecne_vs_text", "spolecne_vs_obraz"]].plot(
    kind="bar", figsize=(max(12, len(ids)*0.35), 5)
)
ax.set_title("Ke které samostatné reprezentaci je společný embedding blíž?")
ax.set_ylabel("Kosinová podobnost")
ax.set_xlabel("Karta")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

## 12. Vizuální mapa všech embeddingů

Pro přehled promítneme textové, obrazové a společné vektory do dvou rozměrů pomocí PCA. PCA zachovává jen část informace, proto je mapa orientační. Všechny tři typy vektorů promítáme společně, ne každou skupinu zvlášť.

Tím lze vizuálně sledovat, zda se reprezentace jedné karty drží pohromadě, nebo se text, obraz a společný vstup v mapě rozcházejí.

In [ ]:
# 12. PCA mapa vektorů
all_vectors = np.vstack([text_vectors, image_vectors, joint_vectors])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(all_vectors)

n = len(ids)
plot_rows = []
for j, modality in enumerate(["text", "obraz", "společně"]):
    for i, card_id in enumerate(ids):
        plot_rows.append({
            "karta": card_id,
            "modalita": modality,
            "PC1": coords[j*n+i, 0],
            "PC2": coords[j*n+i, 1],
        })
pca_df = pd.DataFrame(plot_rows)

plt.figure(figsize=(13, 10))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="modalita", style="modalita", s=95)
for _, row in pca_df.iterrows():
    plt.annotate(row["karta"], (row["PC1"], row["PC2"]), fontsize=7, alpha=0.75)
plt.title("PCA · text, obraz a společné embeddingy")
plt.tight_layout()
plt.show()

pca_df.to_csv(OUTPUT_DIR / "pca_coordinates.csv", index=False, encoding="utf-8-sig")
print("Podíl vysvětlené variance:", pca.explained_variance_ratio_.round(4),
      "celkem:", round(float(pca.explained_variance_ratio_.sum()), 4))

## 13. Přehled rozsahu tří textů

Tento přehled uvádí délku pohádek po odstranění technických hlaviček a úvodního rámce před názvem. Není to skóre kvality ani významové hustoty.

In [ ]:
# 13. Přehled rozsahu textu (pouze kontext, nikoli skóre kvality)
length_df = pd.DataFrame([
    {
        "karta": r["id"],
        "znaky_vcetne_mezer": len(r["text"]),
        "slova_podle_mezer": len(r["text"].split()),
        "pocet_odstavcu": len([p for p in re.split(r"\n\s*\n", r["text"]) if p.strip()]),
    }
    for r in records
])
display(length_df.sort_values("znaky_vcetne_mezer"))
length_df.to_csv(OUTPUT_DIR / "text_length_context.csv", index=False, encoding="utf-8-sig")

## 14. Export všech dat

ZIP bude obsahovat:
- surové embeddingy všech tří režimů;
- metadata a původní texty;
- matice podobností;
- pořadí správných ilustrací;
- srovnání společných a samostatných reprezentací;
- souřadnice PCA;
- přehled rozsahu textů;
- kopie použitých ilustrací.

Tohle je pracovní datový balík pro další analýzu v MOUSE.

In [ ]:
# 14. Zabalení výsledků
# Uložíme i obrázky použité v experimentu, aby export zůstal dohledatelný.
images_out = OUTPUT_DIR / "images"
images_out.mkdir(exist_ok=True)
for r in records:
    with open(images_out / r["image_file"], "wb") as f:
        f.write(r["image_bytes"])

# Uložíme stručný manifest experimentu.
manifest = {
    "experiment": "MOUSE Multimodal Embeddings v1",
    "model": MODEL_ID,
    "language": "Czech",
    "records_loaded": len(records),
    "modalities": ["text_only", "image_only", "text_plus_image"],
    "text_source": f"{REPO}/{TEXT_DIR}/",
    "image_source": f"{REPO}/{IMAGE_DIR}/",
    "failed_records": errors,
    "notes": [
        "No external fairy-tale references are included.",
        "No English translations are included.",
        "No MOUSE angle scores or semantic-density scores are fabricated.",
        "Cosine similarity describes model representation geometry, not human-validated meaning."
    ]
}
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

zip_path = Path("/kaggle/working/MOUSE_Multimodal_Embeddings_v1_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(OUTPUT_DIR.parent))

print("Hotovo:", zip_path)
print("Velikost MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

print("ZIP je uložen v /kaggle/working. V panelu Output ho můžeš stáhnout.")

## 15. Jak číst výsledky

**Text × obraz heatmapa:** ukazuje, zda se správné texty a ilustrace v modelovém prostoru párují lépe než nesprávné kombinace. Důležitá je diagonála a pořadí správné ilustrace.

**Textová, obrazová a společná matice:** ukazují tři různé struktury podobnosti. Rozdíly mezi nimi jsou podnětem k dalšímu zkoumání, nikoli automatickým důkazem významové vrstvy.

**Společná reprezentace:** ukazuje, jak se reprezentace mění, když model dostane oba vstupy. Není to prostý součet textového a obrazového vektoru.

**PCA mapa:** je orientační projekce do dvou rozměrů. Většina informace z původních vektorů se do ní nemusí vejít.

### Co tento notebook zatím netvrdí
- že měří skutečný počet významů;
- že automaticky rozpoznává 12 úhlů MOUSE;
- že vysoká podobnost znamená správnou interpretaci;
- že výsledek dokazuje účinnost MOUSE.

### Co z něj získáme
První společný soubor textových, obrazových a multimodálních reprezentací vlastních karet, jejich vzájemné podobnosti a vizuální mapu. To je základ pro další práci s vrstvami, uzly a úhly MOUSE.